# 🍕 WB14 — Tableau Executive Intelligence Layer

Transform cleaned Pizza House order data into a Tableau-ready analytical dataset designed for executive dashboards, recruiter demonstrations, Tableau Public projects, GitHub integration, and LinkedIn portfolio assets.

---

## Objectives

• Prepare a Tableau-optimized analytical dataset

• Engineer business-focused features developed throughout the project

• Enable interactive executive storytelling

• Support recruiter-ready dashboard demonstrations

• Extend the Pizza House Sales Analysis project into Tableau

---

## Deliverables

1. orders_2025_tableau.csv

2. Tableau Executive Dashboard

3. Tableau Public portfolio pieces

4. GitHub integration

5. LinkedIn Featured integration

## 14.1 Load Quarterly Source Files

Load the original quarterly Pizza House order files to preserve the full operational dataset before preparing the Tableau export layer.

In [2]:
import pandas as pd

q1 = pd.read_csv("../data/raw/order_items_2025_q1.csv")
q2 = pd.read_csv("../data/raw/order_items_2025_q2.csv")
q3 = pd.read_csv("../data/raw/order_items_2025_q3.csv")
q4 = pd.read_csv("../data/raw/order_items_2025_q4.csv")

orders_raw = pd.concat([q1, q2, q3, q4], ignore_index=True)

print("Dataset Shape:", orders_raw.shape)

orders_raw.head()

Dataset Shape: (50487, 25)


,Order Date,Order ID,Invoice Number,Order Number,Order Type,Order Employee ID,Order Employee Name,Order Employee Custom ID,Note,Currency,...,Order Total,Payments Total,Payment Note,Refunds Total,Refund Credit Surcharge,Manual Refunds Total,Tender,Credit Card Auth Code,Credit Card Transaction ID,Order Payment State
0,31-Mar-2025 10:54 PM PDT,QQBMWE3NMZQ0Y,NaN,19.0,Pick Up,BTCYVG7SFZR5R,Oleg,NaN,Vernon to go Pick up time: 11:09 PM,USD,...,25.99,25.99,NaN,0.0,0.0,0.0,Cash,NaN,NaN,Paid
1,31-Mar-2025 10:35 PM PDT,TH041BMRDC2Y6,NaN,78.0,Delivery,BTCYVG7SFZR5R,Oleg,NaN,Pick up time: 11:35 PM,USD,...,25.13,0.00,NaN,0.0,0.0,0.0,NaN,NaN,NaN,Open
2,31-Mar-2025 10:33 PM PDT,06KJV691ZMV48,NaN,77.0,Pick Up,BTCYVG7SFZR5R,Oleg,NaN,Pick up time: 11:03 PM,USD,...,17.90,17.90,NaN,0.0,0.0,0.0,Debit Card,000376,509100536853,Paid
3,31-Mar-2025 10:24 PM PDT,ASKA2CPN7GT2A,NaN,76.0,Pick Up,BTCYVG7SFZR5R,Oleg,NaN,Queen Pick up time: 10:54 PM\nGenerated reward...,USD,...,40.69,40.69,NaN,0.0,0.0,0.0,Cash,NaN,NaN,Paid
4,31-Mar-2025 10:23 PM PDT,GK5Z5D128NA1M,NaN,75.0,Pick Up,BTCYVG7SFZR5R,Oleg,NaN,Jose Pick up time: 10:53 PM,USD,...,20.05,20.05,NaN,0.0,0.0,0.0,Credit Card,08630D,509100536870,Paid


### 14.1.1 Review Source Dataset Structure

Review the preserved source columns before reshaping for Tableau.

In [3]:
orders_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50487 entries, 0 to 50486
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Order Date                  50487 non-null  object 
 1   Order ID                    50487 non-null  object 
 2   Invoice Number              1301 non-null   object 
 3   Order Number                45028 non-null  float64
 4   Order Type                  50486 non-null  object 
 5   Order Employee ID           50487 non-null  object 
 6   Order Employee Name         50487 non-null  object 
 7   Order Employee Custom ID    0 non-null      float64
 8   Note                        42213 non-null  object 
 9   Currency                    50487 non-null  object 
 10  Tax Amount                  50487 non-null  float64
 11  Tip                         50487 non-null  float64
 12  Service Charge              2 non-null      float64
 13  Credit Surcharge            504

In [4]:
orders_raw.columns.tolist()

['Order Date',
 'Order ID',
 'Invoice Number',
 'Order Number',
 'Order Type',
 'Order Employee ID',
 'Order Employee Name',
 'Order Employee Custom ID',
 'Note',
 'Currency',
 'Tax Amount',
 'Tip',
 'Service Charge',
 'Credit Surcharge',
 'Discount',
 'Order Total',
 'Payments Total',
 'Payment Note',
 'Refunds Total',
 'Refund Credit Surcharge',
 'Manual Refunds Total',
 'Tender',
 'Credit Card Auth Code',
 'Credit Card Transaction ID',
 'Order Payment State']

## 14.2 Convert Date Fields

Convert Clover order timestamps into datetime format while preserving Pacific Time reporting consistency.

In [9]:
orders_raw["Order Date"] = (
    orders_raw["Order Date"]
    .str.replace(" PDT", "", regex=False)
    .str.replace(" PST", "", regex=False)
)

orders_raw["Order Date"] = pd.to_datetime(
    orders_raw["Order Date"],
    format="%d-%b-%Y %I:%M %p",
    errors="coerce"
)

print("Missing Order Date values:", orders_raw["Order Date"].isna().sum())

Missing Order Date values: 0


In [10]:
print("Earliest Order:", orders_raw["Order Date"].min())
print("Latest Order:", orders_raw["Order Date"].max())

Earliest Order: 2025-01-01 11:03:00
Latest Order: 2025-12-31 22:02:00


## 14.3 Engineer Time Intelligence Features

Create time-based analytical fields developed throughout previous workbooks to support Tableau filtering, trend analysis, operational reporting, and executive storytelling.

In [11]:
orders_raw["order_date"] = orders_raw["Order Date"].dt.date

orders_raw["order_hour"] = orders_raw["Order Date"].dt.hour

orders_raw["order_dow"] = (
    orders_raw["Order Date"]
    .dt.day_name()
)

orders_raw["month"] = (
    orders_raw["Order Date"]
    .dt.month_name()
)

orders_raw["year"] = (
    orders_raw["Order Date"]
    .dt.year
)

### 14.3.1 Validate Time Intelligence Features

Review engineered time fields prior to additional business feature development.

In [12]:
orders_raw[
    [
        "Order Date",
        "order_date",
        "order_hour",
        "order_dow",
        "month",
        "year"
    ]
].head()

,Order Date,order_date,order_hour,order_dow,month,year
0,2025-03-31 22:54:00,2025-03-31,22,Monday,March,2025
1,2025-03-31 22:35:00,2025-03-31,22,Monday,March,2025
2,2025-03-31 22:33:00,2025-03-31,22,Monday,March,2025
3,2025-03-31 22:24:00,2025-03-31,22,Monday,March,2025
4,2025-03-31 22:23:00,2025-03-31,22,Monday,March,2025


## 14.4 Create Weekend Indicator

Create a weekend classification feature to support operational comparisons and executive reporting between weekday and weekend performance.

In [13]:
orders_raw["is_weekend"] = (
    orders_raw["order_dow"]
    .isin(["Saturday", "Sunday"])
)

### 14.4.1 Validate Weekend Classification

Review the weekend indicator to ensure accurate assignment across the reporting period.

In [14]:
orders_raw[
    [
        "order_dow",
        "is_weekend"
    ]
].drop_duplicates().sort_values(
    by="order_dow"
)

,order_dow,is_weekend
475,Friday,False
0,Monday,False
279,Saturday,True
114,Sunday,True
655,Thursday,False
871,Tuesday,False
774,Wednesday,False


## 14.5 Create Order Value Segments

Create revenue-based order segments developed throughout previous workbooks to support customer behavior analysis, revenue concentration studies, and executive dashboard storytelling.

In [15]:
orders_raw["order_segment"] = pd.cut(
    orders_raw["Order Total"],
    bins=[0, 15, 30, 45, float("inf")],
    labels=[
        "Low",
        "Mid",
        "High",
        "Very High"
    ],
    right=False
)

### 14.5.1 Validate Order Segments

Review the distribution of order value segments prior to Tableau export.

In [16]:
orders_raw["order_segment"].value_counts()

order_segment
Mid          15215
High         14956
Low          11369
Very High     8947
Name: count, dtype: int64

### 14.5.2 Validate Segment Logic

Confirm that segment assignments align with the defined business rules.

In [17]:
orders_raw[
    [
        "Order Total",
        "order_segment"
    ]
].sample(
    10,
    random_state=42
)

,Order Total,order_segment
23182,25.99,Mid
8230,42.34,High
50025,23.77,Mid
18995,29.97,Mid
20524,37.19,High
8409,72.46,Very High
44600,9.15,Low
1030,0.00,Low
24632,45.84,Very High
42942,18.25,Mid


## 14.6 Create Trend Analysis Fields

Create supporting fields for chronological analysis and Tableau dashboard development.

In [18]:
orders_raw["month_num"] = (
    orders_raw["Order Date"]
    .dt.month
)

orders_raw["quarter"] = (
    orders_raw["Order Date"]
    .dt.quarter
)

### 14.6.1 Validate Trend Fields

Review chronological fields prior to Tableau export.

In [19]:
orders_raw[
    [
        "Order Date",
        "month",
        "month_num",
        "quarter"
    ]
].head()

,Order Date,month,month_num,quarter
0,2025-03-31 22:54:00,March,3,1
1,2025-03-31 22:35:00,March,3,1
2,2025-03-31 22:33:00,March,3,1
3,2025-03-31 22:24:00,March,3,1
4,2025-03-31 22:23:00,March,3,1


## 14.7 Export Tableau Dataset

Prepare and export the final Tableau Intelligence Layer dataset for dashboard development and executive reporting.

In [21]:
tableau_fields = [
    "Order ID",
    "Order Date",
    "Order Type",
    "Tax Amount",
    "Tip",
    "Order Total",
    "Payments Total",
    "Tender",
    "order_date",
    "order_hour",
    "order_dow",
    "is_weekend",
    "month",
    "month_num",
    "quarter",
    "year",
    "order_segment"
]

orders_tableau = orders_raw[
    tableau_fields
].copy()

orders_tableau.to_csv(
    "../data/cleaned/orders_2025_tableau.csv",
    index=False
)

print("Export Shape:", orders_tableau.shape)

Export Shape: (50487, 17)


## 14.8 Validate Tableau Dataset

Perform a final validation of the Tableau Intelligence Layer to ensure the exported dataset is complete, consistent, and ready for dashboard development.

In [22]:
print("Tableau Dataset Shape:")

orders_tableau.shape

Tableau Dataset Shape:


(50487, 17)

### 14.8.1 Review Dataset Structure

Confirm field names, data types, and overall dataset integrity.

In [24]:
orders_tableau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50487 entries, 0 to 50486
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        50487 non-null  object        
 1   Order Date      50487 non-null  datetime64[ns]
 2   Order Type      50486 non-null  object        
 3   Tax Amount      50487 non-null  float64       
 4   Tip             50487 non-null  float64       
 5   Order Total     50487 non-null  float64       
 6   Payments Total  50487 non-null  float64       
 7   Tender          43379 non-null  object        
 8   order_date      50487 non-null  object        
 9   order_hour      50487 non-null  int32         
 10  order_dow       50487 non-null  object        
 11  is_weekend      50487 non-null  bool          
 12  month           50487 non-null  object        
 13  month_num       50487 non-null  int32         
 14  quarter         50487 non-null  int32         
 15  ye

### 14.8.2 Preview Tableau Dataset

Review the first several records prior to Tableau import.

In [25]:
orders_tableau.head()

,Order ID,Order Date,Order Type,Tax Amount,Tip,Order Total,Payments Total,Tender,order_date,order_hour,order_dow,is_weekend,month,month_num,quarter,year,order_segment
0,QQBMWE3NMZQ0Y,2025-03-31 22:54:00,Pick Up,2.09,0.00,25.99,25.99,Cash,2025-03-31,22,Monday,False,March,3,1,2025,Mid
1,TH041BMRDC2Y6,2025-03-31 22:35:00,Delivery,0.00,0.00,25.13,0.00,NaN,2025-03-31,22,Monday,False,March,3,1,2025,Mid
2,06KJV691ZMV48,2025-03-31 22:33:00,Pick Up,1.44,3.29,17.90,17.90,Debit Card,2025-03-31,22,Monday,False,March,3,1,2025,Mid
3,ASKA2CPN7GT2A,2025-03-31 22:24:00,Pick Up,3.27,0.00,40.69,40.69,Cash,2025-03-31,22,Monday,False,March,3,1,2025,High
4,GK5Z5D128NA1M,2025-03-31 22:23:00,Pick Up,1.61,0.92,20.05,20.05,Credit Card,2025-03-31,22,Monday,False,March,3,1,2025,Mid


### 14.8.3 Check for Missing Values

Identify any remaining null values that may affect Tableau dashboards.

In [26]:
orders_tableau.isnull().sum()

Order ID             0
Order Date           0
Order Type           1
Tax Amount           0
Tip                  0
Order Total          0
Payments Total       0
Tender            7108
order_date           0
order_hour           0
order_dow            0
is_weekend           0
month                0
month_num            0
quarter              0
year                 0
order_segment        0
dtype: int64

### 14.8.4 Validate Order Segment Distribution

Confirm that revenue segmentation was preserved during export.

In [27]:
orders_tableau["order_segment"].value_counts()

order_segment
Mid          15215
High         14956
Low          11369
Very High     8947
Name: count, dtype: int64

### 14.8.5 Validate Weekend Classification

Confirm the distribution of weekday and weekend transactions.

In [28]:
orders_tableau["is_weekend"].value_counts()

is_weekend
False    33044
True     17443
Name: count, dtype: int64

### 14.8.6 Validate Order Type Distribution

Review operational channel composition prior to dashboard development.

In [29]:
orders_tableau["Order Type"].value_counts()

Order Type
Pick Up             33733
Delivery            11299
In-store Pickup      2925
Popmenu Pickup       1743
Popmenu Delivery      786
Name: count, dtype: int64

### 14.8.7 Confirm Tableau Export Location

Verify that the Tableau Intelligence Layer dataset has been successfully exported.

In [30]:
print(
    "Tableau dataset successfully prepared:",
    "../data/cleaned/orders_2025_tableau.csv"
)

Tableau dataset successfully prepared: ../data/cleaned/orders_2025_tableau.csv


### WB14 Summary

Successfully transformed raw Clover POS exports into a Tableau-ready Intelligence Layer by integrating time intelligence, revenue segmentation, operational indicators, and trend analysis fields. The resulting dataset supports executive dashboard development, Tableau Public portfolio projects, recruiter demonstrations, and GitHub integration.